# Categorical EDA — Student Performance

**Author:** Mehruza Nosirova

This notebook analyzes the **categorical variables** of the cleaned UCI Student Performance dataset (`student-por-clean.csv`, 649 students, 33 columns).

Steps:
1. Identify the categorical variables and group them by type
2. Build a summary table for all categorical variables
3. Value counts (count and percent) for every variable
4. Compare the final grade `G3` across categories
5. Short observations

Note: some numeric columns (`Medu`, `Fedu`, `traveltime`, `studytime`, `famrel`, `freetime`, `goout`, `Dalc`, `Walc`, `health`) are ordinal scales coded as numbers. They are covered in the Numerical EDA, so this notebook focuses on the 17 text-coded categorical variables.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from my_modules.categorical_eda_module import (
    get_categorical_columns,
    categorical_summary,
    value_counts_table,
    target_by_category,
    category_gap_ranking,
)

In [2]:
df = pd.read_csv("../data/student-por-clean.csv")
df.shape

(649, 33)

## 1. Categorical variables

The dataset has 17 categorical variables. They fall into three groups:
- **Binary yes/no**: support, activities and lifestyle flags
- **Binary with codes**: school, sex, address, family size, parents' cohabitation status
- **Nominal with 3+ categories**: parents' jobs, reason for choosing the school, guardian

In [3]:
cat_cols = get_categorical_columns(df)

binary_yes_no = [c for c in cat_cols if set(df[c].unique()) == {"yes", "no"}]
binary_other = [c for c in cat_cols if df[c].nunique() == 2 and c not in binary_yes_no]
multi_category = [c for c in cat_cols if df[c].nunique() > 2]

print("Number of categorical variables:", len(cat_cols))
print("Binary yes/no:", binary_yes_no)
print("Binary (other codes):", binary_other)
print("3+ categories:", multi_category)

Number of categorical variables: 17
Binary yes/no: ['schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Binary (other codes): ['school', 'sex', 'address', 'famsize', 'Pstatus']
3+ categories: ['Mjob', 'Fjob', 'reason', 'guardian']


## 2. Summary table

One row per variable: number of categories, the categories themselves, the most frequent category and its share, and missing values.

In [4]:
summary = categorical_summary(df, cat_cols)
summary

,n_unique,categories,most_frequent,most_frequent_pct,missing_pct
variable,,,,,
school,2,"GP, MS",GP,65.2,0.0
sex,2,"F, M",F,59.0,0.0
address,2,"R, U",U,69.6,0.0
famsize,2,"GT3, LE3",GT3,70.4,0.0
Pstatus,2,"A, T",T,87.7,0.0
Mjob,5,"at_home, health, other, services, teacher",other,39.8,0.0
Fjob,5,"at_home, health, other, services, teacher",other,56.5,0.0
reason,4,"course, home, other, reputation",course,43.9,0.0
guardian,3,"father, mother, other",mother,70.1,0.0


## 3. Value counts

Count and percent of students in each category.

In [5]:
for col in binary_other + multi_category:
    print(f"=== {col} ===")
    print(value_counts_table(df, col))
    print()

=== school ===
        count  percent
school                
GP        423     65.2
MS        226     34.8

=== sex ===
     count  percent
sex                
F      383     59.0
M      266     41.0

=== address ===
         count  percent
address                
U          452     69.6
R          197     30.4

=== famsize ===
         count  percent
famsize                
GT3        457     70.4
LE3        192     29.6

=== Pstatus ===
         count  percent
Pstatus                
T          569     87.7
A           80     12.3

=== Mjob ===
          count  percent
Mjob                    
other       258     39.8
services    136     21.0
at_home     135     20.8
teacher      72     11.1
health       48      7.4

=== Fjob ===
          count  percent
Fjob                    
other       367     56.5
services    181     27.9
at_home      42      6.5
teacher      36      5.5
health       23      3.5

=== reason ===
            count  percent
reason                    
course       

Yes/no variables are easier to compare side by side, so they are combined into one table showing the share of students who answered **yes**.

In [6]:
yes_share = pd.DataFrame({
    "yes_count": [(df[c] == "yes").sum() for c in binary_yes_no],
    "yes_pct": [round((df[c] == "yes").mean() * 100, 1) for c in binary_yes_no],
}, index=binary_yes_no).sort_values("yes_pct", ascending=False)

yes_share

,yes_count,yes_pct
higher,580,89.4
nursery,521,80.3
internet,498,76.7
famsup,398,61.3
activities,315,48.5
romantic,239,36.8
schoolsup,68,10.5
paid,39,6.0


## 4. Final grade (G3) by category

`G3` is the target variable (final grade, 0–20). For each category we show the number of students, mean and median `G3`, and `fail_pct` — the percent of students with `G3 < 10` (below the passing grade in the Portuguese system).

Overall reference values:

In [7]:
print("Overall mean G3:", round(df["G3"].mean(), 2))
print("Overall median G3:", df["G3"].median())
print("Overall fail % (G3 < 10):", round((df["G3"] < 10).mean() * 100, 1))

Overall mean G3: 11.91
Overall median G3: 12.0
Overall fail % (G3 < 10): 15.4


In [8]:
for col in cat_cols:
    print(f"=== {col} ===")
    print(target_by_category(df, col))
    print()

=== school ===
        count  mean_G3  median_G3  fail_pct
school                                     
GP        423    12.58       13.0       7.6
MS        226    10.65       11.0      30.1

=== sex ===
     count  mean_G3  median_G3  fail_pct
sex                                     
F      383    12.25       12.0      13.1
M      266    11.41       11.0      18.8

=== address ===
         count  mean_G3  median_G3  fail_pct
address                                     
U          452    12.26       12.0      12.4
R          197    11.09       11.0      22.3

=== famsize ===
         count  mean_G3  median_G3  fail_pct
famsize                                     
LE3        192    12.13       12.0      12.5
GT3        457    11.81       12.0      16.6

=== Pstatus ===
         count  mean_G3  median_G3  fail_pct
Pstatus                                     
A           80    11.91       12.0      15.0
T          569    11.91       12.0      15.5

=== Mjob ===
          count  mean_G3  m

### Ranking: which variables show the largest difference in G3?

For every variable, the gap between the category with the highest mean `G3` and the category with the lowest mean `G3`.

In [9]:
gap_table = category_gap_ranking(df, cat_cols)
gap_table

,highest_category,highest_mean_G3,lowest_category,lowest_mean_G3,gap
variable,,,,,
higher,yes,12.28,no,8.80,3.48
reason,reputation,12.94,other,10.69,2.25
Fjob,teacher,13.58,at_home,11.43,2.15
Mjob,teacher,13.14,at_home,11.04,2.09
school,GP,12.58,MS,10.65,1.93
guardian,father,12.20,other,10.90,1.30
address,U,12.26,R,11.09,1.18
internet,yes,12.17,no,11.03,1.15
sex,F,12.25,M,11.41,0.85


## 5. Observations

**Distribution of categories**
- The data is **imbalanced** for several variables: 89–90% of students want higher education (`higher`), do not receive extra school support (`schoolsup`), and 94% do not take paid extra classes (`paid`). Small groups (39–69 students) make comparisons less reliable.
- About two thirds of students attend **Gabriel Pereira (GP)**, 59% are **female**, 70% live in **urban** areas, and 88% have parents living together (`Pstatus = T`).
- `Mjob` and `Fjob` are dominated by the vague category **"other"** (40% and 57%), which limits how much these variables can tell us.
- The **mother** is the guardian for 70% of students. `activities` is the only nearly balanced variable (48.5% yes).
- No missing values in any categorical variable.

**Relationship with the final grade (G3)**
- **`higher`** shows the strongest difference: students who do **not** plan higher education average 8.8 vs 12.3, and 48% of them fail compared with 12% of the others.
- **`school`**: MS students average about 2 points lower than GP students (10.7 vs 12.6) and fail four times as often (30% vs 8%).
- **`Fjob` / `Mjob`**: children of **teachers** have the highest averages (13.6 and 13.1); children of mothers working **at home** have the lowest among mothers' jobs (11.0).
- **`reason`**: students who chose the school for its **reputation** perform best (12.9, 7% fail); those with reason "other" perform worst (10.7, 25% fail).
- `address`, `internet`, `sex`, `romantic` and `guardian` show moderate gaps (≈0.6–1.3 points): urban, internet access, female, not in a relationship → slightly higher grades.
- `Pstatus`, `nursery`, `famsize`, `activities` and `famsup` show almost **no difference** in G3.
- Interesting: students with extra school support (`schoolsup`) or paid classes (`paid`) have slightly **lower** averages — likely because these supports go to students who were already struggling.

These are preliminary, descriptive findings — they show association, not causation. The strongest candidates for further analysis are `higher`, `school`, `Fjob`/`Mjob`, `reason`, `address` and `internet`.